## Apache Iceberg

Apache Iceberg é um formato de tabela para Data Lakes que permite operações ACID
(INSERT, UPDATE, DELETE) diretamente sobre arquivos em storage distribuído.

Neste notebook:
- Utilizamos o catálogo `local` com namespace `default`
- Criamos uma tabela Iceberg (`policies`)
- Demonstramos operações de INSERT, UPDATE e DELETE via Apache Spark

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IcebergLocalDevelopment")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1"
    )
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.local",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.local.type",
        "hadoop"
    )
    .config(
        "spark.sql.catalog.local.warehouse",
        "file:///home/william/APACHE-SPARK-COM-DELTA-LAKE-E-APACHE-ICEBERG/iceberg/spark-warehouse"
    )
    .getOrCreate()
)

your 131072x1 screen size is bogus. expect trouble
26/04/26 22:27:26 WARN Utils: Your hostname, DESKTOP-J27O5E0 resolves to a loopback address: 127.0.1.1; using 172.22.146.234 instead (on interface eth0)
26/04/26 22:27:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/william/.cache/pypoetry/virtualenvs/spark-iceberg-41Jidkke-py3.11/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/william/.ivy2/cache
The jars for the packages stored in: /home/william/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ae988288-d558-482c-83ae-f80258179f60;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 in central
:: resolution report :: resolve 122ms :: artifacts dl 3ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.6.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   1   |   0   |   0   |   0   ||   1   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.

In [2]:
spark.sql("DROP TABLE IF EXISTS local.policies")

DataFrame[]

In [3]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.default")

DataFrame[]

In [4]:
spark.sql("SHOW NAMESPACES IN local").show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [5]:
spark.sql("""
CREATE TABLE local.default.policies (
  policy_id INT,
  customer_name STRING,
  policy_type STRING,
  premium_value DECIMAL(10,2),
  status STRING,
  created_at TIMESTAMP
)
USING ICEBERG
""")

DataFrame[]

In [6]:
spark.sql("""
INSERT INTO local.default.policies VALUES
(1, 'Ana Silva', 'AUTO', 120.50, 'ACTIVE', current_timestamp()),
(2, 'Carlos Lima', 'HOME', 300.00, 'ACTIVE', current_timestamp())
""")

spark.sql("""
SELECT * FROM local.default.policies
""").show(truncate=False)

+---------+-------------+-----------+-------------+------+--------------------------+
|policy_id|customer_name|policy_type|premium_value|status|created_at                |
+---------+-------------+-----------+-------------+------+--------------------------+
|1        |Ana Silva    |AUTO       |120.50       |ACTIVE|2026-04-26 22:27:51.317057|
|2        |Carlos Lima  |HOME       |300.00       |ACTIVE|2026-04-26 22:27:51.317057|
+---------+-------------+-----------+-------------+------+--------------------------+



In [7]:
spark.sql("""
UPDATE local.default.policies
SET status = 'CANCELED'
WHERE policy_id = 1
""")

spark.sql("""
SELECT * FROM local.default.policies
""").show(truncate=False)

+---------+-------------+-----------+-------------+--------+--------------------------+
|policy_id|customer_name|policy_type|premium_value|status  |created_at                |
+---------+-------------+-----------+-------------+--------+--------------------------+
|2        |Carlos Lima  |HOME       |300.00       |ACTIVE  |2026-04-26 22:27:51.317057|
|1        |Ana Silva    |AUTO       |120.50       |CANCELED|2026-04-26 22:27:51.317057|
+---------+-------------+-----------+-------------+--------+--------------------------+



In [8]:
spark.sql("""
DELETE FROM local.default.policies
WHERE policy_id = 2
""")

spark.sql("""
SELECT * FROM local.default.policies
""").show(truncate=False)

+---------+-------------+-----------+-------------+--------+--------------------------+
|policy_id|customer_name|policy_type|premium_value|status  |created_at                |
+---------+-------------+-----------+-------------+--------+--------------------------+
|1        |Ana Silva    |AUTO       |120.50       |CANCELED|2026-04-26 22:27:51.317057|
+---------+-------------+-----------+-------------+--------+--------------------------+



In [9]:
spark.sql("SHOW TABLES IN local.default").show()


+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|  default| policies|      false|
+---------+---------+-----------+



In [10]:
spark.sql("DESCRIBE TABLE local.default.policies").show()

+-------------+-------------+-------+
|     col_name|    data_type|comment|
+-------------+-------------+-------+
|    policy_id|          int|   NULL|
|customer_name|       string|   NULL|
|  policy_type|       string|   NULL|
|premium_value|decimal(10,2)|   NULL|
|       status|       string|   NULL|
|   created_at|    timestamp|   NULL|
+-------------+-------------+-------+

